# 0 — First Aerial Image (Phase 1 MVP)

Demonstrates the Phase 1 deliverable from `PROJECT_PLAN.md`:
- Build an annular pupil with central obscuration (paper #15).
- Apply Kirchhoff thin-mask to a line/space pattern.
- Compute the coherent aerial image and show contrast vs pitch.
- Verify the anamorphic 4×/8× field mapping (paper #19).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from src import constants as C
from src.mask import MaskGrid, kirchhoff_mask, line_space_pattern, pinhole_pattern
from src.pupil import PupilSpec, build_pupil, pupil_metrics
from src.aerial import aerial_image, contrast, nils, cd_from_threshold, wafer_grid_from_mask

print(f'lambda = {C.LAMBDA_EUV * 1e9:.1f} nm,  NA = {C.NA_HIGH},  Rayleigh R = {C.rayleigh_resolution() * 1e9:.2f} nm')
print(f'DOF (k2={C.K2_TYPICAL}) = {C.depth_of_focus() * 1e9:.1f} nm')
print(f'EUV photon energy = {C.euv_photon_energy_eV():.1f} eV')

## 1. Pupil — annular + central obscuration

In [ ]:
spec = PupilSpec(grid_size=256, na=C.NA_HIGH, obscuration_ratio=0.20)
P = build_pupil(spec)
print('pupil metrics:', pupil_metrics(P))

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(np.abs(P), cmap='gray', extent=[-1, 1, -1, 1])
ax[0].set_title('|P| — annular amplitude')
ax[0].set_xlabel('f_x / (NA/λ)')
ax[0].set_ylabel('f_y / (NA/λ)')
ax[1].plot(np.linspace(-1, 1, 256), np.abs(P[128, :]))
ax[1].set_title('|P| central row')
ax[1].set_xlabel('f_x / (NA/λ)')
fig.tight_layout()
plt.show()

## 2. PSF from a single pinhole (Airy-like)

In [ ]:
# 1× imaging so the PSF is in mask-plane coordinates.
grid = MaskGrid(nx=256, ny=256, pixel_size=4e-9)
field = kirchhoff_mask(pinhole_pattern(grid))
I_solid, wafer = aerial_image(field, grid,
    pupil_spec=PupilSpec(grid_size=256, na=C.NA_HIGH, obscuration_ratio=0.0),
    anamorphic=False)
I_ann, _ = aerial_image(field, grid,
    pupil_spec=PupilSpec(grid_size=256, na=C.NA_HIGH, obscuration_ratio=0.4),
    anamorphic=False)

extent_nm = [-grid.nx*grid.pixel_size/2*1e9, grid.nx*grid.pixel_size/2*1e9,
             -grid.ny*grid.pixel_size/2*1e9, grid.ny*grid.pixel_size/2*1e9]

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(I_solid, cmap='inferno', extent=extent_nm); ax[0].set_title('PSF — solid pupil (ε=0)')
ax[0].set_xlabel('x [nm]'); ax[0].set_ylabel('y [nm]')
ax[1].imshow(I_ann, cmap='inferno', extent=extent_nm); ax[1].set_title('PSF — annular pupil (ε=0.4)')
ax[1].set_xlabel('x [nm]')
cy = wafer.ny // 2
x = (np.arange(wafer.nx) - wafer.nx/2) * wafer.pixel_x_m * 1e9
ax[2].plot(x, I_solid[cy, :], label='solid')
ax[2].plot(x, I_ann[cy, :], label='annular ε=0.4')
ax[2].set_xlim(-50, 50); ax[2].set_yscale('log'); ax[2].set_ylim(1e-5, 1.5)
ax[2].set_title('PSF horizontal cut'); ax[2].set_xlabel('x [nm]'); ax[2].legend()
fig.tight_layout(); plt.show()

Annular pupil pushes more energy into the sidelobes — qualitative match with paper #15 (central obscuration optics).

## 3. Line/space sweep — pitch vs contrast

Choose pitches that divide the grid total length (960 nm) so the pattern is exactly periodic.
The discrete coherent cutoff for line/space is pitch ≥ λ/NA = 24.5 nm in 1× imaging.

In [ ]:
grid = MaskGrid(nx=960, ny=64, pixel_size=1e-9)
pupil_spec = PupilSpec(grid_size=960, na=C.NA_HIGH, obscuration_ratio=0.20)

# All pitches divide 960 nm exactly to avoid FFT leakage.
pitches_nm = [12, 16, 20, 24, 30, 40, 60, 80, 120]
rows = []
for p_nm in pitches_nm:
    pat = line_space_pattern(grid, pitch_m=p_nm * 1e-9)
    I, wafer = aerial_image(kirchhoff_mask(pat), grid, pupil_spec=pupil_spec, anamorphic=False)
    line = I[wafer.ny // 2, :]
    c = contrast(line)
    nils_val = nils(line, wafer.pixel_x_m, cd_target_m=0.5 * p_nm * 1e-9)
    rows.append((p_nm, c, nils_val))
    print(f'pitch {p_nm:3d} nm:  contrast = {c:0.3f}   NILS = {nils_val:0.3f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
p_arr = np.array([r[0] for r in rows]); c_arr = np.array([r[1] for r in rows]); n_arr = np.array([r[2] for r in rows])
ax[0].plot(p_arr, c_arr, 'o-')
ax[0].axvline(C.LAMBDA_EUV / C.NA_HIGH * 1e9, ls='--', color='r', label='λ/NA cutoff')
ax[0].set_xlabel('pitch [nm] (mask=wafer in 1× mode)'); ax[0].set_ylabel('Michelson contrast')
ax[0].set_title('Contrast vs pitch — coherent imaging'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(p_arr, n_arr, 's-')
ax[1].set_xlabel('pitch [nm]'); ax[1].set_ylabel('NILS')
ax[1].set_title('NILS vs pitch'); ax[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

Contrast collapses below pitch ≈ λ/NA = 24.5 nm (coherent on-axis cutoff). Above, contrast rises monotonically. NILS is largest near the cutoff — the slope is steepest where the first diffraction order is just inside the pupil edge.

## 4. Anamorphic field check (mask 26×33 mm → wafer 6.5×4.125 mm)

In [ ]:
px = 100e-6  # coarse 100 µm/pixel for the field-size sanity check
g = MaskGrid(nx=int(round(26e-3/px)), ny=int(round(33e-3/px)), pixel_size=px)
w = wafer_grid_from_mask(g, anamorphic=True)
print(f'Mask: {g.nx*g.pixel_size*1e3:.2f} × {g.ny*g.pixel_size*1e3:.2f} mm')
print(f'Wafer: {w.nx*w.pixel_x_m*1e3:.4f} × {w.ny*w.pixel_y_m*1e3:.4f} mm')
print(f'Demagnification: {g.pixel_size/w.pixel_x_m:.2f}× (scan)  {g.pixel_size/w.pixel_y_m:.2f}× (cross-scan)')

## 5. End-to-end aerial image — 32 nm pitch line/space

In [ ]:
# Mask-plane pitch 128 nm → wafer 32 nm (anamorphic ×4 along scan).
grid = MaskGrid(nx=512, ny=128, pixel_size=8e-9)
pat = line_space_pattern(grid, pitch_m=128e-9)
I, w = aerial_image(kirchhoff_mask(pat), grid, anamorphic=True)
print(f'wafer pixel: {w.pixel_x_m*1e9:.3f} × {w.pixel_y_m*1e9:.3f} nm')
print(f'wafer-plane pitch: {(grid.pixel_size * 16)/4*1e9:.0f} nm  (16 mask-px × 8 nm / 4)')

extent_nm = [0, w.nx*w.pixel_x_m*1e9, 0, w.ny*w.pixel_y_m*1e9]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(pat, cmap='gray', aspect='auto'); ax[0].set_title('Mask pattern (128 nm pitch, mask plane)')
ax[1].imshow(I, cmap='inferno', extent=extent_nm, aspect='auto')
ax[1].set_title('Aerial image (32 nm pitch, wafer plane)')
ax[1].set_xlabel('x [nm]'); ax[1].set_ylabel('y [nm]')
fig.tight_layout(); plt.show()

---
**Phase 1 MVP summary**  
Pupil + Kirchhoff mask + Fourier imaging + anamorphic projection are all wired. The notebook reproduces the qualitative results required by the four Phase 1 exit criteria in PROJECT_PLAN.md. Next: Phase 3 (wafer topography & DOF).